# Amber Advanced Tutorial : MM-PBSA Alanine Scanning
## Binding Free Energy of the Ras-Raf Complex

This notebook automates the [Amber MM-PBSA Alanine Scanning Tutorial](https://ambermd.org/tutorials/advanced/tutorial3/py_script/section3.php). 
It is configured to run AmberTools commands inside an **Apptainer** container (.sif).

### Prerequisites
* Ensure **Apptainer** is installed.
* Update the `SIF_PATH` below to point to the project container file.
* Place `ras-raf.pdb`, `ras.pdb`, and `raf.pdb` in your working directory.

In [1]:
import os
import subprocess

# --- CONFIGURE THIS PATH ---
SIF_PATH = "amber_ready.sif"

def run_apptainer(cmd):
    full_cmd = f"wsl apptainer exec {SIF_PATH} {cmd}"
    print(f"Executing: {full_cmd}")

    result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
        print("❌ COMMAND FAILED!")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
    else:
        print("✅ Success!")
        print(result.stdout)

# Verify the container exists
if not os.path.exists(SIF_PATH):
    print(f"Warning: Container not found at {SIF_PATH}. Please update the path.")

# --- DOWNLOAD INPUT PDB FILES ---
import urllib.request

pdb_files = {
    "ras-raf.pdb": "https://ambermd.org/tutorials/advanced/tutorial3/py_script/files/ras-raf.pdb",
    "ras.pdb":     "https://ambermd.org/tutorials/advanced/tutorial3/py_script/files/ras.pdb",
}

for filename, url in pdb_files.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"✅ {filename} downloaded")
    else:
        print(f"✅ {filename} already exists, skipping")

✅ ras-raf.pdb already exists, skipping
✅ ras.pdb already exists, skipping


## Section 1: Prepare the Mutant PDB Files

We will mutate residue 21 (Isoleucine, I21) to Alanine in both `ras-raf.pdb` and `ras.pdb`.
This requires removing all side chain atoms beyond the beta-carbon (CB) and renaming the residue from `ILE` to `ALA`.

In [2]:
# Atoms to remove for I21 -> ALA mutation (lines for atoms 294-305 in the pdb)
ATOMS_TO_REMOVE = ["CG1", "CG2", "CD1"]

def make_ala_mutant(input_pdb, output_pdb, residue_num=21):
    """Mutates a given ILE residue to ALA by removing side chain atoms beyond CB."""
    with open(input_pdb, 'r') as f:
        lines = f.readlines()

    mutant_lines = []
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            res_num = int(line[22:26].strip())
            atom_name = line[12:16].strip()
            res_name = line[17:20].strip()
            # Skip side chain atoms beyond CB for the target ILE residue
            if res_num == residue_num and res_name == "ILE" and atom_name in ATOMS_TO_REMOVE:
                continue
            # Rename ILE -> ALA for the target residue
            if res_num == residue_num and res_name == "ILE":
                line = line[:17] + "ALA" + line[20:]
        mutant_lines.append(line)

    with open(output_pdb, 'w') as f:
        f.writelines(mutant_lines)

    print(f"✅ Mutant PDB written to {output_pdb}")

make_ala_mutant("ras-raf.pdb", "ras-raf_mutant.pdb", residue_num=21)
make_ala_mutant("ras.pdb",     "ras_mutant.pdb",     residue_num=21)

✅ Mutant PDB written to ras-raf_mutant.pdb
✅ Mutant PDB written to ras_mutant.pdb


## Section 2: Build Topology and Coordinate Files with tleap

Using tleap we will create `.prmtop` and `.inpcrd` files for the non-mutant complex, receptor, and ligand,
the solvated complex for MD, and the mutant structures.

In [3]:
tleap_input = """
source leaprc.protein.ff19SB
source leaprc.water.tip3p

com = loadpdb ras-raf.pdb
ras = loadpdb ras.pdb
raf = loadpdb raf.pdb

saveamberparm com ras-raf.prmtop ras-raf.inpcrd
saveamberparm ras ras.prmtop ras.inpcrd
saveamberparm raf raf.prmtop raf.inpcrd

charge com
solvatebox com TIP3PBOX 12.0
saveamberparm com ras-raf_solvated.prmtop ras-raf_solvated.inpcrd

com_mut = loadpdb ras-raf_mutant.pdb
ras_mut = loadpdb ras_mutant.pdb
saveamberparm com_mut rasraf_mutant.prmtop rasraf_mutant.inpcrd
saveamberparm ras_mut ras_mutant.prmtop ras_mutant.inpcrd

quit
"""

with open("tleap.in", "w") as f:
    f.write(tleap_input)

print("✅ tleap.in written")
run_apptainer('bash -c "cd /mnt/c/Users/Swag3/Downloads/amber69/data && tleap -s -f tleap.in"')

✅ tleap.in written
Executing: wsl apptainer exec amber_ready.sif bash -c "cd /mnt/c/Users/Swag3/Downloads/amber69/data && tleap -s -f tleap.in"
❌ COMMAND FAILED!
STDOUT: 
STDERR: /bin/sh: apptainer: not found



## Section 3: Run MMPBSA.py Alanine Scanning

We will calculate the binding free energy using both MM-GBSA and MM-PBSA methods on the wild-type
and mutant (I21A) structures. Results are written to `FINAL_RESULTS_MMPBSA.dat`.

In [ ]:
mmpbsa_input = """\
sample input file for running alanine scanning
 &general
   startframe=1, endframe=50, interval=1,
   verbose=1,
/
&gb
  saltcon=0.1
/
&pb
  istrng=0.100
/
&alanine_scanning
/
"""

with open("mmpbsa.in", "w") as f:
    f.write(mmpbsa_input)

print("✅ mmpbsa.in written")

run_apptainer(
    "bash -c 'cd /mnt/c/Users/Swag3/Downloads/amber69/data && MMPBSA.py -O "
    "-i mmpbsa.in "
    "-sp ras-raf_solvated.prmtop "
    "-cp ras-raf.prmtop "
    "-rp ras.prmtop "
    "-lp raf.prmtop "
    "-y *.mdcrd "
    "-mc rasraf_mutant.prmtop "
    "-mr ras_mutant.prmtop'"
)

✅ mmpbsa.in written
Executing: apptainer exec amber_ready.sif MMPBSA.py -O -i mmpbsa.in -sp ras-raf_solvated.prmtop -cp ras-raf.prmtop -rp ras.prmtop -lp raf.prmtop -y *.mdcrd -mc rasraf_mutant.prmtop -mr ras_mutant.prmtop
❌ COMMAND FAILED!
STDOUT: 
STDERR: 'apptainer' is not recognized as an internal or external command,
operable program or batch file.



## Section 4: View Results

Print the final binding free energy results from `FINAL_RESULTS_MMPBSA.dat`.

In [9]:
results_file = "FINAL_RESULTS_MMPBSA.dat"

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        print(f.read())
else:
    print(f"Warning: {results_file} not found. Ensure MMPBSA.py ran successfully.")